# 08 - Figuras del paper (formato Springer LNCS)

Regenera las 7 figuras del manuscrito cumpliendo las *Instructions for Authors*:

- salida **vectorial** en `.pdf` (mas un `.png` a 400 dpi de respaldo)
- lettering **>= 6 pt** una vez insertada la figura en el `.tex`
- **legible en escala de grises** (el impreso es en blanco y negro): paleta gris + hatching
- el barrido de theta se acota a `[0, 0.60]`, que es hasta donde hay predicciones del CNN

No reentrena nada: solo lee `results/models/*.pkl` y `results/features/*.csv`.
Correr desde `notebooks/`.

In [1]:
import numpy as np
import pandas as pd
import joblib
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix

ROOT = Path('..')
MODELS = ROOT / 'results' / 'models'
FEATS = ROOT / 'results' / 'features'
OUT = ROOT / 'results' / 'figures' / 'paper'
OUT.mkdir(parents=True, exist_ok=True)

THETA = 0.60

# Costo del front end clasico (segmentacion + 113 features + SVM) relativo al costo
# de una inferencia MobileNetV2. Se despeja de la ecuacion (7) del paper con el punto
# de operacion reportado: 0.44 = K_FRONT + 0.258.
K_FRONT = 0.182

# Ancho de texto de LNCS en pulgadas. Sirve para verificar el minimo de 6 pt.
LNCS_TEXTWIDTH_IN = 4.8

plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 110,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
})

REGISTRO = []

def guardar(fig, nombre, ancho_tex=1.0, min_pt=9):
    fig.savefig(OUT / f'{nombre}.pdf')
    fig.savefig(OUT / f'{nombre}.png', dpi=400)
    ancho_in = fig.get_size_inches()[0]
    escala = (LNCS_TEXTWIDTH_IN * ancho_tex) / ancho_in
    REGISTRO.append({
        'figura': nombre,
        'ancho_fig_in': round(ancho_in, 2),
        'ancho_tex': ancho_tex,
        'escala': round(escala, 3),
        'min_pt_fuente': min_pt,
        'pt_efectivo': round(min_pt * escala, 2),
        'ok_6pt': min_pt * escala >= 6.0,
    })
    plt.close(fig)
    print('guardado', nombre)

## Carga de predicciones

In [2]:
y_test = np.asarray(joblib.load(MODELS / 'y_test.pkl'))
pred_svm = np.asarray(joblib.load(MODELS / 'pred_svm.pkl'))
pred_hibrido = np.asarray(joblib.load(MODELS / 'pred_hibrido.pkl'))
confianza = np.asarray(joblib.load(MODELS / 'confianza.pkl'))
fuente = np.asarray(joblib.load(MODELS / 'fuente.pkl'))
clases = sorted(np.unique(y_test))

print('test:', len(y_test), 'imagenes,', len(clases), 'clases')
print('accuracy SVM     : %.3f' % accuracy_score(y_test, pred_svm))
print('accuracy hibrido : %.3f' % accuracy_score(y_test, pred_hibrido))
print('escalado al CNN  : %.1f%% (%d imagenes)' % (100 * (fuente == 'CNN').mean(), (fuente == 'CNN').sum()))

test: 2328 imagenes, 9 clases
accuracy SVM     : 0.780
accuracy hibrido : 0.891
escalado al CNN  : 25.8% (601 imagenes)


## Baseline de minima distancia

Se reconstruye desde los prototipos precalculados para poder graficar el recall de las
tres vias en la Fig. 3.

In [3]:
proto = np.load(MODELS / 'prototipos_min_distancia.npz', allow_pickle=True)
feat_cols = [str(c) for c in proto['feat_cols']]
mu, sigma = proto['mu'], proto['sigma']
prototipos = proto['prototipos']
clases_proto = np.array([str(c) for c in proto['clases']])

df_test = pd.read_csv(FEATS / 'features_test.csv')
X = df_test[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(float)

Z = (X - mu) / sigma
P = (prototipos - mu) / sigma
d2 = ((Z[:, None, :] - P[None, :, :]) ** 2).sum(axis=2)
pred_min = clases_proto[np.argmin(d2, axis=1)]

print('accuracy min-distance : %.3f' % accuracy_score(y_test, pred_min))
print('macro-F1 min-distance : %.3f' % f1_score(y_test, pred_min, average='macro'))

accuracy min-distance : 0.537
macro-F1 min-distance : 0.438


## Fig. 1 - Distribucion de confianza del SVM (validacion)

In [4]:
svm = joblib.load(MODELS / 'svm_clasico.pkl')
scaler = joblib.load(MODELS / 'scaler_clasico.pkl')
cols_svm = joblib.load(MODELS / 'features_clasico.pkl')

df_val = pd.read_csv(FEATS / 'features_val.csv')
X_val = df_val[cols_svm].replace([np.inf, -np.inf], np.nan).fillna(0)
if 'forma_aspect_ratio' in X_val.columns:
    X_val['forma_aspect_ratio'] = X_val['forma_aspect_ratio'].clip(upper=5)

conf_val = svm.predict_proba(scaler.transform(X_val)).max(axis=1)

fig, ax = plt.subplots(figsize=(6.0, 2.6))
ax.hist(conf_val, bins=40, color='0.45', edgecolor='white', linewidth=0.4)
ax.axvline(THETA, color='k', linestyle='--', linewidth=1.3, label=r'$\theta$ = 0.60')
ax.set_xlabel('SVM maximum class probability')
ax.set_ylabel('Frequency')
ax.legend(frameon=False)
guardar(fig, 'fig12_svm_confidence_dist', ancho_tex=0.9, min_pt=9)

guardado fig12_svm_confidence_dist


## Fig. 2 - F1 por clase, SVM vs hibrido

In [5]:
f1_svm = f1_score(y_test, pred_svm, average=None, labels=clases)
f1_hib = f1_score(y_test, pred_hibrido, average=None, labels=clases)

x = np.arange(len(clases))
w = 0.38

fig, ax = plt.subplots(figsize=(6.4, 2.9))
ax.bar(x - w / 2, f1_svm, w, label='SVM-RBF', color='0.72', edgecolor='k', linewidth=0.5)
ax.bar(x + w / 2, f1_hib, w, label='Hybrid (SVM + CNN)', color='0.35', edgecolor='k', linewidth=0.5, hatch='//')
ax.set_xticks(x)
ax.set_xticklabels(clases, rotation=30, ha='right')
ax.set_ylabel('F1-score')
ax.set_ylim(0, 1)
ax.legend(frameon=False, ncol=2, loc='upper center', bbox_to_anchor=(0.5, 1.18))
guardar(fig, 'fig10_f1_by_class', ancho_tex=1.0, min_pt=9)

guardado fig10_f1_by_class


## Fig. 3 - Recall por clase, clasico vs SVM vs hibrido

In [6]:
rec_min = recall_score(y_test, pred_min, average=None, labels=clases)
rec_svm = recall_score(y_test, pred_svm, average=None, labels=clases)
rec_hib = recall_score(y_test, pred_hibrido, average=None, labels=clases)

w = 0.27

fig, ax = plt.subplots(figsize=(6.4, 2.9))
ax.bar(x - w, rec_min, w, label='Min-distance', color='0.85', edgecolor='k', linewidth=0.5)
ax.bar(x, rec_svm, w, label='SVM-RBF', color='0.60', edgecolor='k', linewidth=0.5, hatch='//')
ax.bar(x + w, rec_hib, w, label='Hybrid', color='0.28', edgecolor='k', linewidth=0.5, hatch='xx')
ax.set_xticks(x)
ax.set_xticklabels(clases, rotation=30, ha='right')
ax.set_ylabel('Recall')
ax.set_ylim(0, 1)
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.18))
guardar(fig, 'fig11_recall_by_class', ancho_tex=1.0, min_pt=9)

guardado fig11_recall_by_class


## Fig. 4 - Tasa de escalado por clase vs ganancia en F1

In [16]:
escalado = np.array([100 * (fuente[y_test == c] == 'CNN').mean() for c in clases])
ganancia = f1_hib - f1_svm

fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.scatter(escalado, ganancia, s=26, color='k')

corte = escalado.min() + 0.7 * (escalado.max() - escalado.min())
for c, a, b in zip(clases, escalado, ganancia):
    if a > corte:
        ax.annotate(c, (a, b), xytext=(-6, 4), textcoords='offset points',
                    fontsize=9, ha='right')
    else:
        ax.annotate(c, (a, b), xytext=(5, 3), textcoords='offset points',
                    fontsize=9, ha='left')

ax.set_xlabel('images escalated to CNN (%)')
ax.set_ylabel(r'$\Delta$F1 (hybrid $-$ SVM)')
ax.set_xlim(18, 46)
ax.set_ylim(0.045, 0.20)
ax.grid(alpha=0.3, linewidth=0.5)
guardar(fig, 'fig16_escalation_vs_gain', ancho_tex=0.9, min_pt=9)

for c, a, b in sorted(zip(clases, escalado, ganancia), key=lambda t: -t[1]):
    print('%-10s escala %.1f%%   dF1 %+.3f' % (c, a, b))

guardado fig16_escalation_vs_gain
metal      escala 40.9%   dF1 +0.181
cardboard  escala 31.3%   dF1 +0.144
glass      escala 31.1%   dF1 +0.110
battery    escala 27.5%   dF1 +0.175
plastic    escala 26.2%   dF1 +0.158
textile    escala 23.7%   dF1 +0.081
paper      escala 21.0%   dF1 +0.061
organic    escala 20.9%   dF1 +0.121
trash      escala 20.2%   dF1 +0.103


## Fig. 5 y Fig. 6 - Barrido de theta y costo

El barrido se acota a `[0, 0.60]`. Por encima de ese valor no hay prediccion del CNN
para las imagenes en `[0.60, theta)`, y sustituirla por la del SVM reproduce la curva
del SVM, lo que aparentaria una meseta que nunca se midio.

In [8]:
thetas = np.arange(0.0, THETA + 1e-9, 0.02)
acc, mf1, tasa = [], [], []
for t in thetas:
    pred_t = np.where(confianza >= t, pred_svm, pred_hibrido)
    acc.append(accuracy_score(y_test, pred_t))
    mf1.append(f1_score(y_test, pred_t, average='macro'))
    tasa.append((confianza < t).mean())

acc, mf1, tasa = np.array(acc), np.array(mf1), np.array(tasa)

fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.7))
axes[0].plot(thetas, acc, 'o-', ms=3, color='k', label='accuracy')
axes[0].plot(thetas, mf1, 's--', ms=3, color='0.55', label='macro-F1')
axes[0].set_xlabel(r'confidence threshold $\theta$')
axes[0].set_ylabel('score')
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.3, linewidth=0.5)
axes[1].plot(100 * tasa, acc, 'o-', ms=3, color='k')
axes[1].set_xlabel('images escalated to CNN (%)')
axes[1].set_ylabel('accuracy')
axes[1].grid(alpha=0.3, linewidth=0.5)
fig.tight_layout()
guardar(fig, 'fig15_theta_sweep', ancho_tex=1.0, min_pt=9)

guardado fig15_theta_sweep


In [9]:
costo = 100 * (K_FRONT + tasa)

fig, ax = plt.subplots(figsize=(5.2, 3.0))
ax.plot(costo, acc, 'o-', ms=3.5, color='k')
ax.set_xlabel('compute per image, % of full-CNN pipeline')
ax.set_ylabel('accuracy')
ax.grid(alpha=0.3, linewidth=0.5)
guardar(fig, 'fig14_accuracy_vs_cost', ancho_tex=0.8, min_pt=9)

print('costo relativo: %.1f%% (theta=0) a %.1f%% (theta=0.60)' % (costo.min(), costo.max()))

guardado fig14_accuracy_vs_cost
costo relativo: 18.2% (theta=0) a 44.0% (theta=0.60)


## Fig. 7 - Matrices de confusion

Los conteos van a 6.5 pt en una figura de 6.6 in insertada a `\linewidth`, lo que da
~4.7 pt efectivos y **no cumple** el minimo de Springer. Por eso se genera tambien una
version apilada, donde cada panel ocupa el ancho completo y los conteos quedan sobre
9 pt. Usa la apilada si el CFP no da problema con cambiar el caption.

In [15]:
def dibujar_cm(ax, pred, titulo, fs_tick, fs_cell, fs_lab):
    cm = confusion_matrix(y_test, pred, labels=clases)
    cmn = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cmn, cmap='Greys', vmin=0, vmax=1)
    ax.set_xticks(range(len(clases)))
    ax.set_yticks(range(len(clases)))
    ax.set_xticklabels(clases, rotation=45, ha='right', fontsize=fs_tick)
    ax.set_yticklabels(clases, fontsize=fs_tick)
    ax.set_xlabel('Predicted label', fontsize=fs_lab)
    ax.set_ylabel('True label', fontsize=fs_lab)
    ax.set_title(titulo, fontsize=fs_lab)
    for i in range(len(clases)):
        for j in range(len(clases)):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=fs_cell,
                        color='white' if cmn[i, j] > 0.5 else 'black')

titulos = ['SVM-RBF (accuracy = 78.0%)', 'Hybrid SVM + CNN (accuracy = 89.1%)']

# version lado a lado (la que espera el .tex actual)
fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.3))
for ax, pred, t in zip(axes, [pred_svm, pred_hibrido], titulos):
    dibujar_cm(ax, pred, t, 7, 6.5, 8)
fig.tight_layout()
guardar(fig, 'fig13_confusion_svm_vs_hybrid', ancho_tex=1.0, min_pt=6.5)

# version apilada, compacta: cumple el minimo de 6 pt sin ocupar la pagina entera
fig, axes = plt.subplots(2, 1, figsize=(3.4, 6.6))
for ax, pred, t in zip(axes, [pred_svm, pred_hibrido], titulos):
    dibujar_cm(ax, pred, t, 8, 7, 8.5)
fig.tight_layout(h_pad=1.2)
guardar(fig, 'fig13_confusion_stacked', ancho_tex=0.65, min_pt=7)

guardado fig13_confusion_svm_vs_hybrid
guardado fig13_confusion_stacked


## Verificacion del minimo de 6 pt

In [11]:
chequeo = pd.DataFrame(REGISTRO)
print(chequeo.to_string(index=False))

fallan = chequeo.loc[~chequeo['ok_6pt'], 'figura'].tolist()
if fallan:
    print('\nNo cumplen el minimo de 6 pt:', ', '.join(fallan))
else:
    print('\nTodas cumplen el minimo de 6 pt.')

                       figura  ancho_fig_in  ancho_tex  escala  min_pt_fuente  pt_efectivo  ok_6pt
    fig12_svm_confidence_dist           6.0        0.9   0.720            9.0         6.48    True
            fig10_f1_by_class           6.4        1.0   0.750            9.0         6.75    True
        fig11_recall_by_class           6.4        1.0   0.750            9.0         6.75    True
     fig16_escalation_vs_gain           5.6        0.9   0.771            9.0         6.94    True
            fig15_theta_sweep           6.6        1.0   0.727            9.0         6.55    True
       fig14_accuracy_vs_cost           5.2        0.8   0.738            9.0         6.65    True
fig13_confusion_svm_vs_hybrid           6.6        1.0   0.727            6.5         4.73   False
      fig13_confusion_stacked           5.0        1.0   0.960            9.0         8.64    True

No cumplen el minimo de 6 pt: fig13_confusion_svm_vs_hybrid


## Copiar a la carpeta del manuscrito

El `.tex` usa `\graphicspath{{figs/}}` y `\DeclareGraphicsExtensions{.pdf,...}`, asi que
basta copiar los `.pdf` a `figs/` y LaTeX los tomara antes que cualquier `.png`.

In [12]:
import shutil

DESTINO = ROOT / 'paper' / 'figs'   # ajusta si tu .tex vive en otro lado
DESTINO.mkdir(parents=True, exist_ok=True)

for pdf in sorted(OUT.glob('*.pdf')):
    shutil.copy2(pdf, DESTINO / pdf.name)
    print('->', DESTINO / pdf.name)

-> ..\paper\figs\fig10_f1_by_class.pdf
-> ..\paper\figs\fig11_recall_by_class.pdf
-> ..\paper\figs\fig12_svm_confidence_dist.pdf
-> ..\paper\figs\fig13_confusion_stacked.pdf
-> ..\paper\figs\fig13_confusion_svm_vs_hybrid.pdf
-> ..\paper\figs\fig14_accuracy_vs_cost.pdf
-> ..\paper\figs\fig15_theta_sweep.pdf
-> ..\paper\figs\fig16_escalation_vs_gain.pdf
